# Continuous Sign Language Recognition on PHOENIX-2014

Multimodal CSLR: two frozen streams (I3D video + HRNet keypoints) → TCN encoders → concat fusion → BiLSTM → CTC → WER.

This notebook runs the full Part II pipeline end to end:
**setup → vocabulary → frame staging → I3D features → keypoint features → train E1–E4 → test evaluation → stride-2 resolution study → demo & figures.**

All paths and hyperparameters live in `config.yaml` (the single source of truth). Every long job is resumable: feature dumps skip already-written files and training checkpoints `best.pt`/`last.pt` each epoch, so a Colab disconnect costs at most one epoch.

**Headline result:** keypoints beat generic video features (44.71 vs 53.37 test WER); naive fusion underperforms at stride 4, is diagnosed as a temporal-resolution mismatch, and—after re-extracting video at stride 2—fusion wins at **40.61 test WER**, beating the best single stream by 4.10 points.

## Stage 0 — Boot & setup
Check the GPU, mount Drive, fix the config filename, unpack the `src/` code package, and load the config.

In [ ]:
# Cell 1: Check what hardware Colab gave us
import torch
import sys

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("No GPU! Go to Runtime → Change runtime type → GPU, then re-run.")

In [ ]:
# Cell 2: Mount Drive (persistent storage) and set up our project folder
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/sign_project'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(os.path.join(PROJECT_DIR, 'checkpoints'), exist_ok=True)

print("Project folder ready at:", PROJECT_DIR)
print("Contents:", os.listdir(PROJECT_DIR))

In [ ]:
import os
base = '/content/drive/MyDrive/sign_project'
src = os.path.join(base, ' config.yaml')   # with leading space
dst = os.path.join(base, 'config.yaml')    # clean
if os.path.exists(src):
    os.rename(src, dst)
    print("renamed ->", os.listdir(base))
else:
    print("no space-prefixed file found; current:", os.listdir(base))

In [ ]:
import zipfile, shutil, os

ZIP  = '/content/drive/MyDrive/cslr-phoenix-project.zip'
DEST = '/content/drive/MyDrive/sign_project'

# Extract to temp, then lift src/ + notebooks/ into sign_project (beside the data).
tmp = '/content/_pkg_tmp'
if os.path.exists(tmp): shutil.rmtree(tmp)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(tmp)

inner = os.path.join(tmp, 'project')   # the zip's top-level folder is 'project/'
for item in ['src', 'notebooks']:
    s = os.path.join(inner, item)
    d = os.path.join(DEST, item)
    if os.path.exists(d): shutil.rmtree(d)
    shutil.copytree(s, d)

# Deliberately NOT copying config.yaml — keep YOUR edited one with sign_project paths.
print("sign_project now contains:", sorted(os.listdir(DEST)))
print("src/utils.py exists:", os.path.exists(os.path.join(DEST, 'src', 'utils.py')))

In [ ]:
import zipfile, os, yaml

# 1. Pull the complete config from the zip's temp extraction
src_cfg = '/content/_pkg_tmp/project/config.yaml'
if not os.path.exists(src_cfg):
    with zipfile.ZipFile('/content/drive/MyDrive/cslr-phoenix-project.zip') as z:
        z.extractall('/content/_pkg_tmp')

with open(src_cfg) as f:
    cfg = yaml.safe_load(f)
print("Sections from zip config:", list(cfg.keys()))   # should show all 11

# 2. Re-apply ONLY your real paths (the sign_project layout we confirmed)
cfg['paths'].update({
    'drive_root':          '/content/drive/MyDrive',
    'dataset_root':        '/content/drive/MyDrive/sign_project/data/phoenix2014-release',
    'corpus_dir':          'phoenix-2014-multisigner',
    'corrnet_repo':        '/content/CorrNet',
    'corrnet_checkpoint':  '/content/drive/MyDrive/sign_project/checkpoints/corrnet_phoenix2014.pt',
    'hrnet_keypoints_raw': '/content/drive/MyDrive/sign_project/data/keypoints_raw',
    'manifests':           '/content/drive/MyDrive/sign_project/manifests',
    'features_rgb':        '/content/drive/MyDrive/sign_project/features_rgb',
    'features_kp':         '/content/drive/MyDrive/sign_project/features_kp',
    'runs':                '/content/drive/MyDrive/sign_project/runs',
})

# 3. Write the COMPLETE, corrected config back
with open('/content/drive/MyDrive/sign_project/config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)
print("config.yaml restored with all sections + your paths")

## Stage 1 — Vocabulary & manifests
Parse the official corpus, verify split counts (5,672/540/629), build the train-only vocabulary (1,231 glosses), and save manifests + a small smoke split.

In [ ]:
from src.vocab import parse_corpus
from src.utils import save_json, p

items = {s: parse_corpus(cfg, s) for s in cfg['dataset']['splits']}
counts = {s: len(v) for s, v in items.items()}
print('counts:', counts)
exp = cfg['dataset']['expected_counts']
for s in exp:
    assert counts[s] == exp[s], f"{s}: got {counts[s]}, expected {exp[s]} — wrong split/release?"
print('OK — full official PHOENIX-2014 split confirmed.')

In [ ]:
from src.vocab import build_vocab, save_vocab

gloss2id = build_vocab(items['train'], blank_index=cfg['vocab']['blank_index'])
save_vocab(gloss2id, p(cfg, 'manifests') / 'vocab.json')
print('vocab size (excl. blank):', len(gloss2id))

# dev/test glosses unseen in train -> unavoidable recognition errors; just report.
train_set = set(gloss2id)
for s in ('dev', 'test'):
    oov = {g for it in items[s] for g in it['glosses']} - train_set
    print(f'{s}: {len(oov)} OOV gloss types (expected small)')

In [ ]:
# Save manifests + smoke split
import random
for s in cfg['dataset']['splits']:
    save_json(items[s], p(cfg, 'manifests') / f'{s}.json')

random.seed(cfg['project']['seed'])
tr_ids = [it['id'] for it in items['train']]
dv_ids = [it['id'] for it in items['dev']]
random.shuffle(tr_ids); random.shuffle(dv_ids)
save_json(tr_ids[:cfg['dataset']['smoke_train']], p(cfg, 'manifests') / 'smoke_train_ids.json')
save_json(dv_ids[:cfg['dataset']['smoke_dev']],   p(cfg, 'manifests') / 'smoke_dev_ids.json')
print('manifests + smoke split saved to', p(cfg, 'manifests'))

In [ ]:
# Gloss-length stats (feeds the T >= 2L check in NB 04)
import numpy as np
for s in cfg['dataset']['splits']:
    L = np.array([len(it['glosses']) for it in items[s]])
    print(f'{s}: glosses/sentence  min {L.min()}  mean {L.mean():.1f}  max {L.max()}')

## Stage 2 — Stage frames to local SSD
The 56 GB frame archive is extracted to the Colab local SSD (fast, reliable) rather than Drive. One-time, ~27 min. Verify folder counts after.

In [ ]:
import os
# Find the tar.gz wherever it is
for path in ['/content/drive/MyDrive/sign_project/data/phoenix-2014.v3.tar.gz',
             '/content/drive/MyDrive/phoenix-2014.v3.tar.gz']:
    if os.path.exists(path):
        print("FOUND:", path, f"({os.path.getsize(path)/1e9:.1f} GB)")
        break
else:
    import glob
    hits = glob.glob('/content/drive/MyDrive/**/phoenix*.tar.gz', recursive=True)
    print("searched; found:", hits if hits else "NO ARCHIVE ON DRIVE")

In [ ]:
import os, time, subprocess

ARCHIVE = '/content/drive/MyDrive/sign_project/data/phoenix-2014.v3.tar.gz'
LOCAL   = '/content/phoenix_local'
os.makedirs(LOCAL, exist_ok=True)

print("Extracting 56GB to local SSD — faster and reliable. ~15-30 min.\n")
t0 = time.time()
# capture return code so we KNOW if tar succeeded, not just guess
result = subprocess.run(['tar', '-xzf', ARCHIVE, '-C', LOCAL],
                        capture_output=True, text=True)
print(f"tar return code: {result.returncode}  ({'OK' if result.returncode==0 else 'FAILED'})")
if result.stderr:
    print("stderr (first 500 chars):", result.stderr[:500])
print(f"elapsed: {(time.time()-t0)/60:.1f} min")

In [ ]:
import os
LOCAL = '/content/phoenix_local'
LOCAL_FRAMES = f'{LOCAL}/phoenix2014-release/phoenix-2014-multisigner/features/fullFrame-210x260px'

for split in ['train','dev','test']:
    d = os.path.join(LOCAL_FRAMES, split)
    if os.path.isdir(d):
        n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d,x))])
        print(f"{split}: {n} sentence folders")
    else:
        print(f"{split}: DIR MISSING")
# Want: train 5672, dev 540, test 629

## Stage 3 — Video features (frozen I3D-R50, stride 4)
Load I3D, strip the classification head, and slide an 8-frame window (stride 4) over each sentence → one 2,048-d vector per window. Resumable dump to `features_rgb/` (~146 min for train).

In [ ]:
!pip install pytorchvideo -q
import torch
m = torch.hub.load("facebookresearch/pytorchvideo:main", model="i3d_r50", pretrained=True)
print("i3d_r50 loaded OK; type:", type(m).__name__)

In [ ]:
import torch, torch.nn as nn

class I3DExtractor(nn.Module):
    """Frozen I3D-R50 trunk. Input clip [B,3,T,H,W] -> [B,2048] pooled feature."""
    def __init__(self, net):
        super().__init__()
        self.blocks = nn.ModuleList(list(net.blocks[:-1]))  # drop ResNetBasicHead
        for p in self.parameters():
            p.requires_grad_(False)
        self.eval()

    @torch.no_grad()
    def forward(self, clip):
        x = clip
        for blk in self.blocks:
            x = blk(x)                      # -> [B, 2048, t, h, w]
        return x.mean(dim=[2, 3, 4])        # global spatiotemporal pool -> [B, 2048]

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("GPU available:", torch.cuda.is_available())   # MUST be True for the real dump
extractor = I3DExtractor(m).to(device)

# verify: one dummy clip -> [1, 2048]
t = torch.randn(1, 3, 8, 224, 224, device=device)
print("extractor output shape:", extractor(t).shape)   # expect [1, 2048]

In [ ]:
import numpy as np, glob, cv2, torch

WINDOW = 8     # I3D clip length (matches the 8x8 Kinetics weights)
STRIDE = 4     # step between windows; smaller = more timesteps, longer sequences
CROP   = 224   # I3D input size

def load_frames(folder_glob):
    """Load all PNG frames for one sentence -> uint8 array [T, H, W, 3] (RGB)."""
    paths = sorted(glob.glob(folder_glob))
    if not paths:
        raise FileNotFoundError(folder_glob)
    frames = []
    for fp in paths:
        img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        # center crop to 224x224 (210x260 frames -> crop the 224 region)
        # height 260 -> crop 224; width 210 < 224, so resize width up first
        if w < CROP or h < CROP:
            scale = CROP / min(h, w)
            img = cv2.resize(img, (int(round(w*scale)), int(round(h*scale))))
            h, w, _ = img.shape
        top, left = (h - CROP)//2, (w - CROP)//2
        frames.append(img[top:top+CROP, left:left+CROP])
    return np.stack(frames)               # [T, 224, 224, 3]

@torch.no_grad()
def extract_sequence(folder_glob):
    """Sliding-window I3D features for one sentence -> [num_windows, 2048] float16."""
    frames = load_frames(folder_glob).astype(np.float32)        # [T,H,W,3]
    # I3D / Kinetics normalization: mean/std per channel (the slow_r50 recipe)
    mean = np.array([0.45, 0.45, 0.45], dtype=np.float32)
    std  = np.array([0.225, 0.225, 0.225], dtype=np.float32)
    frames = (frames/255.0 - mean) / std
    x = torch.from_numpy(frames).permute(3, 0, 1, 2).to(device)  # [3, T, H, W]
    T = x.shape[1]
    # pad short clips up to one full window
    if T < WINDOW:
        pad = WINDOW - T
        x = torch.cat([x, x[:, -1:].repeat(1, pad, 1, 1)], dim=1)
        T = WINDOW
    feats = []
    for start in range(0, T - WINDOW + 1, STRIDE):
        clip = x[:, start:start+WINDOW].unsqueeze(0)             # [1,3,8,224,224]
        feats.append(extractor(clip).squeeze(0).cpu())          # [2048]
    if not feats:                                                # T==WINDOW exactly
        feats.append(extractor(x.unsqueeze(0)).squeeze(0).cpu())
    return torch.stack(feats).numpy().astype('float16')         # [num_windows, 2048]

# --- test on ONE real sample before dumping anything ---
from src.utils import load_json, p
items = load_json(p(cfg, 'manifests') / 'train.json')
sample = items[0]
print("sample id:", sample['id'])
print("folder field:", sample['folder'])

In [ ]:
import yaml
CFG_PATH = '/content/drive/MyDrive/sign_project/config.yaml'
with open(CFG_PATH) as f:
    c = yaml.safe_load(f)

c['rgb_dump']['feature_dim'] = 2048
c['rgb_dump']['crop'] = 224
c['model']['rgb_in_dim'] = 2048      # the downstream encoder reads this

with open(CFG_PATH, 'w') as f:
    yaml.safe_dump(c, f, sort_keys=False, allow_unicode=True)

# reload so the running session uses the new values
cfg = load_config('config.yaml')
print("rgb_in_dim now:", cfg['model']['rgb_in_dim'])   # expect 2048

In [ ]:
import os, glob
from src.utils import load_json, p

FRAMES_ROOT = f'{LOCAL}/phoenix2014-release/phoenix-2014-multisigner/features/fullFrame-210x260px'

def sample_glob(split, folder_field):
    return os.path.join(FRAMES_ROOT, split, folder_field)

for split in ['train', 'dev', 'test']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    missing = [it['id'] for it in items
               if not os.path.isdir(os.path.join(FRAMES_ROOT, split, it['folder'].replace('/1/*.png','')))]
    print(f"{split}: {len(items)-len(missing)}/{len(items)} have frames, {len(missing)} missing")

In [ ]:
import os, glob, time, numpy as np
from src.utils import load_json, p, save_json

OUT_ROOT = cfg['paths']['features_rgb']

for split in ['train', 'dev', 'test']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    out_dir = os.path.join(OUT_ROOT, split)
    os.makedirs(out_dir, exist_ok=True)
    done = {os.path.splitext(f)[0] for f in os.listdir(out_dir)}
    fail_log = os.path.join(OUT_ROOT, f'failures_{split}.json')
    print(f"\n=== {split}: {len(items)} samples, {len(done)} already done ===")
    t0 = time.time()
    for n, it in enumerate(items):
        if it['id'] in done:
            continue
        try:
            seq = extract_sequence(sample_glob(split, it['folder']))
            np.save(os.path.join(out_dir, f"{it['id']}.npy"), seq)
        except Exception as e:
            print("  FAIL", it['id'], repr(e))
            fails = load_json(fail_log) if os.path.exists(fail_log) else {}
            fails[it['id']] = repr(e); save_json(fails, fail_log)
        if (n+1) % 50 == 0:
            rate = (n+1)/(time.time()-t0)
            print(f"  {split} {n+1}/{len(items)}  ({rate:.1f}/s, ~{(len(items)-n-1)/rate/60:.0f} min left)")
    print(f"{split} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
import os
from src.utils import load_json, p
OUT_ROOT = cfg['paths']['features_rgb']
for split in ['train','dev','test']:
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    n = len([f for f in os.listdir(os.path.join(OUT_ROOT, split)) if f.endswith('.npy')])
    fails = os.path.join(OUT_ROOT, f'failures_{split}.json')
    nf = len(load_json(fails)) if os.path.exists(fails) else 0
    print(f"{split}: {n}/{len(items)} features, {nf} failures")

In [ ]:
import os
os.remove(os.path.join(cfg['paths']['features_rgb'], 'failures_train.json'))
print("stale log removed — RGB features complete and verified: 5672/540/629")

## Stage 4 — Keypoint features (HRNet → 53 points, 265-d)
PHOENIX ships no keypoints; download the pre-extracted HRNet whole-body keypoints (MSKA release), inspect the format, select 53 signing-relevant points, shoulder-normalize, append velocities, standardize with train-only stats, and check CTC feasibility.

In [ ]:
!pip install -q gdown
import gdown
result = gdown.download_folder(
    "https://drive.google.com/drive/folders/1D_iVtqeARBLO7WcZCTGCAdHXkKqHfF9X",
    output="/content/drive/MyDrive/sign_project/data/keypoints_raw",
    quiet=False, use_cookies=False
)
print("RESULT:", result)

In [ ]:
import os, pickle

KP = '/content/drive/MyDrive/sign_project/data/keypoints_raw'

# 1. What ARE these files? Check the magic bytes.
for split in ['dev']:   # start with the small one
    path = os.path.join(KP, f'Phoenix-2014.{split}')
    with open(path, 'rb') as f:
        head = f.read(16)
    print(f"{split}: first 16 bytes = {head}")
    # pickle starts with \x80; npz is 'PK' (zip); numpy .npy is \x93NUMPY; json is '{' or '['

In [ ]:
import os, pickle

KP = '/content/drive/MyDrive/sign_project/data/keypoints_raw'
with open(os.path.join(KP, 'Phoenix-2014.dev'), 'rb') as f:
    data = pickle.load(f)

print("type:", type(data))
if isinstance(data, dict):
    keys = list(data.keys())
    print("num entries:", len(keys), "(dev should be ~540)")
    print("first 3 keys:", keys[:3])
    k = keys[0]
    v = data[k]
    print(f"\nentry '{k}':")
    print("  value type:", type(v))
    if hasattr(v, 'shape'):
        print("  shape:", v.shape, "dtype:", getattr(v, 'dtype', ''))
    elif isinstance(v, dict):
        print("  sub-keys:", list(v.keys()))
        for sk, sv in v.items():
            print(f"    {sk}: {type(sv).__name__}", getattr(sv, 'shape', ''), getattr(sv, 'dtype', ''))
    elif isinstance(v, (list, tuple)):
        print("  length:", len(v), "| first elem:", type(v[0]).__name__, getattr(v[0], 'shape', ''))
else:
    print("not a dict — structure:", str(data)[:200])

In [ ]:
import os, pickle, torch
KP = '/content/drive/MyDrive/sign_project/data/keypoints_raw'
with open(os.path.join(KP, 'Phoenix-2014.dev'), 'rb') as f:
    data = pickle.load(f)

k = list(data.keys())[0]
kp = data[k]['keypoint']          # [T, 133, 3]
print("shape:", kp.shape)
print("name field:", data[k]['name'])
print("num_frames field:", data[k]['num_frames'], "vs keypoint T:", kp.shape[0])

# COCO-WholeBody layout: indices 0-16 body (5=L shoulder, 6=R shoulder),
# 17-22 feet, 23-90 face (68), 91-111 left hand (21), 112-132 right hand (21).
# Sanity-check by looking at the spread of body keypoints in frame 0.
frame0 = kp[0]                    # [133, 3]
print("\nbody keypoints (0-16), [x, y, conf]:")
for i in range(17):
    print(f"  {i}: {frame0[i].tolist()}")

In [ ]:
import os, pickle, torch, numpy as np

KP_RAW = '/content/drive/MyDrive/sign_project/data/keypoints_raw'

# COCO-WholeBody 133-point layout:
#   0-16   body (5=L-shoulder, 6=R-shoulder, 7-8 elbows, 9-10 wrists, 0-4 head)
#   17-22  feet            <- DROP (irrelevant for signing)
#   23-90  face (68)       <- DROP most (noisy); keep a few mouth points optionally
#   91-111 left hand (21)  <- KEEP (core signal)
#   112-132 right hand (21)<- KEEP (core signal)

UPPER_BODY = list(range(0, 11))          # head + shoulders + elbows + wrists (drop 11-16 hips/legs)
LEFT_HAND  = list(range(91, 112))        # 21 points
RIGHT_HAND = list(range(112, 133))       # 21 points
# Optional: a few mouth points from the face block (COCO-WholeBody mouth ~ 71-90)
# MOUTH = list(range(71, 91))  # uncomment to include
SELECTED = UPPER_BODY + LEFT_HAND + RIGHT_HAND   # 11 + 21 + 21 = 53 points

# Shoulder indices WITHIN the original 133 (before selection)
LS_RAW, RS_RAW = 5, 6
# Their positions AFTER selection (they're in UPPER_BODY, so same order)
LS_SEL, RS_SEL = SELECTED.index(5), SELECTED.index(6)

print(f"keeping {len(SELECTED)} of 133 points")
print(f"shoulders at selected indices {LS_SEL}, {RS_SEL}")

# Load all three split pickles
splits_data = {}
for split in ['train', 'dev', 'test']:
    with open(os.path.join(KP_RAW, f'Phoenix-2014.{split}'), 'rb') as f:
        splits_data[split] = pickle.load(f)
    print(f"{split}: {len(splits_data[split])} entries")

def kp_sample_id(raw_key):
    """'dev/01April_..._default-1/1/' -> '01April_..._default-1' (your manifest id)."""
    return raw_key.split('/')[1]

# Verify key mapping matches your manifests
from src.utils import load_json, p
manifest_ids = {it['id'] for it in load_json(p(cfg,'manifests')/'dev.json')}
kp_ids = {kp_sample_id(k) for k in splits_data['dev'].keys()}
print(f"\ndev: {len(kp_ids & manifest_ids)}/{len(manifest_ids)} keypoint IDs match manifest")

In [ ]:
import os, numpy as np, torch
from src.utils import load_json, p, save_json

OUT_KP = cfg['paths']['features_kp']

def featurize(kp_tensor):
    """[T,133,3] raw -> [T, 53*5] features (x,y,conf,dx,dy), shoulder-normalized."""
    kp = kp_tensor.numpy() if torch.is_tensor(kp_tensor) else np.asarray(kp_tensor)
    kp = kp[:, SELECTED, :].astype(np.float32)        # [T, 53, 3] -> select points
    xy   = kp[:, :, :2]                                # [T,53,2]
    conf = kp[:, :, 2:3]                               # [T,53,1]
    # shoulder-centered, shoulder-width-scaled (per-video, stable via median)
    mid   = (xy[:, LS_SEL] + xy[:, RS_SEL]) / 2.0      # [T,2]
    width = np.linalg.norm(xy[:, LS_SEL] - xy[:, RS_SEL], axis=-1, keepdims=True)  # [T,1]
    scale = np.median(width[width > 1e-3]) if np.any(width > 1e-3) else 1.0
    center = np.median(mid, axis=0)                    # [2], per-video anchor
    xy = (xy - center[None, None]) / (scale + 1e-6)
    vel = np.zeros_like(xy); vel[1:] = xy[1:] - xy[:-1]
    feat = np.concatenate([xy, conf, vel], axis=-1)    # [T,53,5]
    return feat.reshape(feat.shape[0], -1).astype(np.float32)   # [T, 265]

# --- Pass 1: train-only mean/std over the feature dimension ---
print("computing train-only normalization stats...")
s = ss = None; cnt = 0
for raw_key, entry in splits_data['train'].items():
    f = featurize(entry['keypoint'])
    s  = f.sum(0) if s  is None else s  + f.sum(0)
    ss = (f**2).sum(0) if ss is None else ss + (f**2).sum(0)
    cnt += len(f)
mean = s / cnt
std  = np.sqrt(np.maximum(ss/cnt - mean**2, 1e-8))
os.makedirs(OUT_KP, exist_ok=True)
save_json({'mean': mean.tolist(), 'std': std.tolist(), 'dim': len(mean),
           'selected_points': SELECTED, 'shoulder_idx': [LS_SEL, RS_SEL]},
          os.path.join(OUT_KP, 'normalization_stats.json'))
print(f"stats saved; feature dim = {len(mean)} (expect 53*5 = 265)")

In [ ]:
# --- Pass 2: standardize + dump every split with the SAME stats ---
for split in ['train', 'dev', 'test']:
    out_dir = os.path.join(OUT_KP, split)
    os.makedirs(out_dir, exist_ok=True)
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    manifest_ids = {it['id'] for it in items}
    n_done = 0
    for raw_key, entry in splits_data[split].items():
        sid = kp_sample_id(raw_key)
        if sid not in manifest_ids:
            continue
        f = (featurize(entry['keypoint']) - mean) / std
        np.save(os.path.join(out_dir, f"{sid}.npy"), f.astype('float16'))
        n_done += 1
    print(f"{split}: {n_done}/{len(items)} keypoint features dumped")

In [ ]:
import yaml
CFG_PATH = '/content/drive/MyDrive/sign_project/config.yaml'
with open(CFG_PATH) as f:
    c = yaml.safe_load(f)

c['model']['kp_in_dim'] = 265      # 53 points x 5 channels (was 395 = old 79x5 assumption)
c['keypoints']['num_points'] = 53  # keep config honest about what we actually used

with open(CFG_PATH, 'w') as f:
    yaml.safe_dump(c, f, sort_keys=False, allow_unicode=True)

cfg = load_config('config.yaml')
print("kp_in_dim now:", cfg['model']['kp_in_dim'])   # expect 265

In [ ]:
import os, numpy as np
from src.utils import load_json, p

OUT_KP = cfg['paths']['features_kp']
infeasible = []
items = load_json(p(cfg, 'manifests') / 'train.json')
for it in items:
    fp = os.path.join(OUT_KP, 'train', f"{it['id']}.npy")
    T = np.load(fp, mmap_mode='r').shape[0]
    if T < 2*len(it['glosses']):
        infeasible.append((it['id'], T, len(it['glosses'])))
print(f"keypoint T<2L infeasible: {len(infeasible)}", infeasible[:5])

# also confirm dim
sample = np.load(os.path.join(OUT_KP, 'dev', os.listdir(os.path.join(OUT_KP,'dev'))[0]))
print("keypoint feature shape:", sample.shape, "dtype:", sample.dtype)

## Stage 5 — Train the stride-4 ladder (E1–E4)
Matched-condition experiment ladder (one variable per row). Each run selects on dev WER and is resumable.

- **E1** video only · **E2** keypoints only · **E3** fusion (concat) · **E4** fusion + auxiliary CTC

After E2, keypoints are resampled down to the RGB length for fusion (`features_kp_aligned/`), and the config is repointed at the aligned set.

In [ ]:
import torch, numpy as np
from torch.utils.data import DataLoader
from src.data import FeatureDataset, collate
from src.models import CSLRModel, count_params
from src.decode import greedy_decode, ids_to_glosses
from src.vocab import load_vocab
from src.utils import load_json, p, get_device

device = get_device()
print("device:", device)
g2i, i2g = load_vocab(p(cfg, 'manifests') / 'vocab.json')
items = load_json(p(cfg, 'manifests') / 'train.json')

# --- T >= 2L feasibility on REAL I3D feature lengths (RGB only) ---
import os
OUT = os.path.join(cfg['paths']['features_rgb'], 'train')
infeasible = []
for it in items:
    fp = os.path.join(OUT, f"{it['id']}.npy")
    T = np.load(fp, mmap_mode='r').shape[0]
    if T < 2*len(it['glosses']):
        infeasible.append((it['id'], T, len(it['glosses'])))
print(f"T<2L infeasible samples: {len(infeasible)}", infeasible[:5])

# --- one forward + CTC + decode on the smoke split, RGB-only ---
smoke = load_json(p(cfg, 'manifests') / 'smoke_train_ids.json')
ds = FeatureDataset(cfg, items, 'train', ('rgb',), g2i, train_mode=True, ids_subset=smoke)
dl = DataLoader(ds, batch_size=4, collate_fn=collate, shuffle=True)
batch = next(iter(dl))
print("batch rgb shape:", tuple(batch['rgb'].shape), "lengths:", batch['lengths'].tolist())

model = CSLRModel(cfg, len(g2i)+1, ('rgb',)).to(device)
print("params:", count_params(model)/1e6, "M")
out = model(batch, device)
crit = torch.nn.CTCLoss(blank=0, zero_infinity=cfg['train']['ctc_zero_infinity'])
loss = crit(out['main'], batch['targets'].to(device),
            out['out_lengths'].to(device), batch['target_lengths'].to(device))
print("CTC loss:", loss.item(), "->", "FINITE OK" if torch.isfinite(loss) else "BAD")
dec = ids_to_glosses(greedy_decode(out['main'], out['out_lengths']), i2g)
print("untrained decode (gibberish expected):", dec[0][:8])

In [ ]:
from src.train import run_training

result = run_training(cfg, streams=("rgb",), run_name="E1_rgb", seed=42)
print("\nBest dev WER:", result['best_dev_wer'])

In [ ]:
from src.train import run_training
result_e2 = run_training(cfg, streams=("kp",), run_name="E2_kp", seed=42)
print("\nE2 keypoint-only best dev WER:", result_e2['best_dev_wer'])
print("(E1 RGB-only baseline was: 53.86%)")

In [ ]:
import os, numpy as np
from src.utils import load_json, p

OUT_KP = cfg['paths']['features_kp']
OUT_RGB = cfg['paths']['features_rgb']
STRIDE = 4   # match RGB's stride

# Re-dump keypoints DOWNSAMPLED to align with RGB length, into a new dir
OUT_KP_ALIGNED = OUT_KP + '_aligned'

for split in ['train','dev','test']:
    in_dir  = os.path.join(OUT_KP, split)
    out_dir = os.path.join(OUT_KP_ALIGNED, split)
    os.makedirs(out_dir, exist_ok=True)
    items = load_json(p(cfg, 'manifests') / f'{split}.json')
    n = 0
    for it in items:
        kp_path  = os.path.join(in_dir, f"{it['id']}.npy")
        rgb_path = os.path.join(OUT_RGB, split, f"{it['id']}.npy")
        if not (os.path.exists(kp_path) and os.path.exists(rgb_path)):
            continue
        kp  = np.load(kp_path)                  # [T_kp, 265] per-frame
        T_rgb = np.load(rgb_path, mmap_mode='r').shape[0]   # target length
        # uniformly sample T_rgb indices from the keypoint sequence
        idx = np.linspace(0, len(kp)-1, T_rgb).round().astype(int)
        kp_aligned = kp[idx]                    # [T_rgb, 265]
        np.save(os.path.join(out_dir, f"{it['id']}.npy"), kp_aligned.astype('float16'))
        n += 1
    print(f"{split}: {n} aligned keypoint features (matched to RGB length)")

In [ ]:
import yaml
CFG_PATH = '/content/drive/MyDrive/sign_project/config.yaml'
with open(CFG_PATH) as f: c = yaml.safe_load(f)
c['paths']['features_kp'] = OUT_KP_ALIGNED   # point fusion at aligned keypoints
with open(CFG_PATH,'w') as f: yaml.safe_dump(c, f, sort_keys=False, allow_unicode=True)
cfg = load_config('config.yaml')
print("features_kp now:", cfg['paths']['features_kp'])

from src.train import run_training
result_e3 = run_training(cfg, streams=("rgb","kp"), run_name="E3_fusion", seed=42)
print("\nE3 fusion best dev WER:", result_e3['best_dev_wer'])
print("E1 RGB-only: 53.86% | E2 keypoint-only: 44.06%")

In [ ]:
from src.train import run_training
result_e4 = run_training(cfg, streams=("rgb","kp"), run_name="E4_aux", seed=42,
                         overrides={"model.aux_ctc": True})
print("\nE4 (fusion + aux CTC) best dev WER:", result_e4['best_dev_wer'])
print("E1: 53.86% | E2: 44.06% | E3: 49.19%")

## Stage 6 — Test evaluation (stride 4)
Score each selected checkpoint once on the held-out test set. Each checkpoint loads its own saved config, so it is evaluated with exactly the feature dirs/dims it was trained on.

Results: E1 53.37 · E2 44.71 · E3 47.86 · E4 45.10 — naive fusion underperforms keypoints-alone (the diagnosed shortfall).

In [ ]:
import torch, os
from src.train import load_model_from_ckpt
from src.data import FeatureDataset, collate
from src.decode import evaluate_greedy
from src.vocab import load_vocab
from src.utils import load_json, p, get_device
from torch.utils.data import DataLoader

device = get_device()
g2i, i2g = load_vocab(p(cfg, 'manifests') / 'vocab.json')

experiments = [
    ("E1_rgb",    ("rgb",)),
    ("E2_kp",     ("kp",)),
    ("E3_fusion", ("rgb","kp")),
    ("E4_aux",    ("rgb","kp")),
]

RUNS = cfg['paths']['runs']
test_items = load_json(p(cfg, 'manifests') / 'test.json')

print(f"{'Exp':<12} {'Streams':<14} {'Dev WER':<10} {'TEST WER':<10}")
print("-"*48)
results = {}
for run_name, streams in experiments:
    ckpt = os.path.join(RUNS, f"{run_name}_seed42", "best.pt")
    if not os.path.exists(ckpt):
        print(f"{run_name}: checkpoint missing, skipping")
        continue
    model, ckpt_cfg, _ = load_model_from_ckpt(ckpt, device)
    ds = FeatureDataset(ckpt_cfg, test_items, 'test', streams, g2i, train_mode=False)
    dl = DataLoader(ds, batch_size=16, collate_fn=collate)
    test_wer, refs, hyps = evaluate_greedy(model, dl, i2g, device)
    dev_best = torch.load(ckpt, map_location='cpu').get('dev_wer', float('nan'))
    results[run_name] = {'test': test_wer['wer'], 'dev': dev_best,
                         'sub': test_wer['sub'], 'ins': test_wer['ins'],
                         'del': test_wer['del'], 'refs': refs, 'hyps': hyps}
    print(f"{run_name:<12} {'+'.join(streams):<14} {dev_best:<10.2f} {test_wer['wer']:<10.2f}")

print("\nTest-set evaluation complete.")

## Stage 7 — Stride-2 resolution study: re-extract video
Diagnosis: aligning per-frame keypoints (~215 steps) down to stride-4 video (~54 steps) discards ~75% of keypoint detail. Fix: re-extract video at **stride 2** (~108 steps), halving the mismatch.

Re-stage frames (runtime cycled), reload I3D, and dump RGB at stride 2 → `features_rgb_stride2/` (~232 min). Verify counts/shapes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
CODE = '/content/drive/MyDrive/sign_project'
sys.path.append(CODE)
%cd $CODE
from src.utils import load_config
cfg = load_config('config.yaml')
print('config loaded:', cfg['project']['name'])

In [ ]:
import os, time, subprocess
ARCHIVE = '/content/drive/MyDrive/sign_project/data/phoenix-2014.v3.tar.gz'
LOCAL = '/content/phoenix_local'
os.makedirs(LOCAL, exist_ok=True)
print("Extracting to local SSD")
t0 = time.time()
r = subprocess.run(['tar','-xzf',ARCHIVE,'-C',LOCAL], capture_output=True, text=True)
print(f"tar exit: {r.returncode} ({'OK' if r.returncode==0 else 'FAILED'}), {(time.time()-t0)/60:.1f} min")
if r.stderr: print("stderr:", r.stderr[:300])

# verify counts
LF = f'{LOCAL}/phoenix2014-release/phoenix-2014-multisigner/features/fullFrame-210x260px'
for s in ['train','dev','test']:
    d = os.path.join(LF, s)
    n = len([x for x in os.listdir(d) if os.path.isdir(os.path.join(d,x))]) if os.path.isdir(d) else 0
    print(f"  {s}: {n} folders")

In [ ]:
import torch, torch.nn as nn
!pip install pytorchvideo -q 2>/dev/null
m = torch.hub.load("facebookresearch/pytorchvideo:main", model="i3d_r50", pretrained=True)

class I3DExtractor(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.blocks = nn.ModuleList(list(net.blocks[:-1]))
        for p in self.parameters(): p.requires_grad_(False)
        self.eval()
    @torch.no_grad()
    def forward(self, clip):
        x = clip
        for blk in self.blocks: x = blk(x)
        return x.mean(dim=[2,3,4])

device = 'cuda' if torch.cuda.is_available() else 'cpu'
extractor = I3DExtractor(m).to(device)
t = torch.randn(1,3,8,224,224, device=device)
print("GPU:", torch.cuda.is_available(), "| extractor out:", extractor(t).shape)

In [ ]:
import os, glob, cv2, numpy as np, time
from src.utils import load_json, p, save_json

WINDOW, STRIDE, CROP = 8, 2, 224   # STRIDE 2 (was 4)
LOCAL = '/content/phoenix_local'
FRAMES_ROOT = f'{LOCAL}/phoenix2014-release/phoenix-2014-multisigner/features/fullFrame-210x260px'
OUT_ROOT = cfg['paths']['features_rgb'] + '_stride2'   # NEW dir, keeps stride-4 intact

def load_frames(folder_glob):
    paths = sorted(glob.glob(folder_glob))
    if not paths: raise FileNotFoundError(folder_glob)
    frames = []
    for fp in paths:
        img = cv2.cvtColor(cv2.imread(fp), cv2.COLOR_BGR2RGB)
        h, w, _ = img.shape
        if w < CROP or h < CROP:
            s = CROP/min(h,w); img = cv2.resize(img,(int(round(w*s)),int(round(h*s)))); h,w,_ = img.shape
        t,l = (h-CROP)//2,(w-CROP)//2
        frames.append(img[t:t+CROP, l:l+CROP])
    return np.stack(frames)

@torch.no_grad()
def extract_sequence(folder_glob):
    frames = load_frames(folder_glob).astype(np.float32)
    frames = (frames/255.0 - np.array([0.45,0.45,0.45],dtype=np.float32))/np.array([0.225,0.225,0.225],dtype=np.float32)
    x = torch.from_numpy(frames).permute(3,0,1,2).to(device)
    T = x.shape[1]
    if T < WINDOW:
        x = torch.cat([x, x[:,-1:].repeat(1,WINDOW-T,1,1)], dim=1); T = WINDOW
    feats = []
    for start in range(0, T-WINDOW+1, STRIDE):
        feats.append(extractor(x[:,start:start+WINDOW].unsqueeze(0)).squeeze(0).cpu())
    if not feats:
        feats.append(extractor(x.unsqueeze(0)).squeeze(0).cpu())
    return torch.stack(feats).numpy().astype('float16')

def sample_glob(split, folder_field):
    return os.path.join(FRAMES_ROOT, split, folder_field)

for split in ['train','dev','test']:
    items = load_json(p(cfg,'manifests')/f'{split}.json')
    out_dir = os.path.join(OUT_ROOT, split); os.makedirs(out_dir, exist_ok=True)
    done = {os.path.splitext(f)[0] for f in os.listdir(out_dir)}
    print(f"\n=== {split}: {len(items)} samples, {len(done)} done ===")
    t0 = time.time()
    for n,it in enumerate(items):
        if it['id'] in done: continue
        try:
            np.save(os.path.join(out_dir,f"{it['id']}.npy"), extract_sequence(sample_glob(split,it['folder'])))
        except Exception as e:
            print("  FAIL", it['id'], repr(e)[:80])
        if (n+1)%50==0:
            r=(n+1)/(time.time()-t0); print(f"  {n+1}/{len(items)} ({r:.1f}/s, ~{(len(items)-n-1)/r/60:.0f} min left)")
    print(f"{split} done in {(time.time()-t0)/60:.1f} min")

In [ ]:
import os, numpy as np
from src.utils import load_json, p
OUT = cfg['paths']['features_rgb'] + '_stride2'
for split in ['train','dev','test']:
    items = load_json(p(cfg,'manifests')/f'{split}.json')
    d = os.path.join(OUT, split)
    n = len([f for f in os.listdir(d) if f.endswith('.npy')])
    sample = np.load(os.path.join(d, os.listdir(d)[0]))
    print(f"{split}: {n}/{len(items)} features, sample shape {sample.shape}")

## Stage 8 — Align keypoints to stride-2
Re-align the full-rate keypoints to the new (longer) stride-2 video lengths → `features_kp_stride2_aligned/`. Verify it persisted before training.

In [ ]:
import os, numpy as np
from src.utils import load_json, p

# source: full-rate keypoints (the un-aligned 53pt/265-d features) — must exist
OUT_KP_FULL = '/content/drive/MyDrive/sign_project/features_kp'
OUT_RGB_S2  = '/content/drive/MyDrive/sign_project/features_rgb_stride2'
OUT_KP_S2   = '/content/drive/MyDrive/sign_project/features_kp_stride2_aligned'

# sanity: confirm the source keypoints exist before we start
src_train = os.path.join(OUT_KP_FULL, 'train')
print("source full-rate keypoints exist:", os.path.isdir(src_train),
      "| files:", len(os.listdir(src_train)) if os.path.isdir(src_train) else 0)

for split in ['train','dev','test']:
    in_dir  = os.path.join(OUT_KP_FULL, split)
    out_dir = os.path.join(OUT_KP_S2, split); os.makedirs(out_dir, exist_ok=True)
    items = load_json(p(cfg,'manifests')/f'{split}.json')
    n = 0
    for it in items:
        kp_path  = os.path.join(in_dir, f"{it['id']}.npy")
        rgb_path = os.path.join(OUT_RGB_S2, split, f"{it['id']}.npy")
        if not (os.path.exists(kp_path) and os.path.exists(rgb_path)):
            continue
        kp = np.load(kp_path)
        T_rgb = np.load(rgb_path, mmap_mode='r').shape[0]
        idx = np.linspace(0, len(kp)-1, T_rgb).round().astype(int)
        np.save(os.path.join(out_dir, f"{it['id']}.npy"), kp[idx].astype('float16'))
        n += 1
    print(f"{split}: {n} aligned keypoint features written")

In [ ]:
import os
d = '/content/drive/MyDrive/sign_project/features_kp_stride2_aligned/train'
print("now exists:", os.path.isdir(d), "| files:", len(os.listdir(d)) if os.path.isdir(d) else 0)

## Stage 9 — Train the stride-2 ladder
Repoint the config at both stride-2 directories, then re-run E1/E3/E4 under `*_s2` names (E2 is resolution-independent and is not re-run).

In [ ]:
import yaml
CFG_PATH = f'{CODE}/config.yaml'
with open(CFG_PATH) as f: c = yaml.safe_load(f)
c['paths']['features_rgb'] = f'{CODE}/features_rgb_stride2'
c['paths']['features_kp']  = f'{CODE}/features_kp_stride2_aligned'
with open(CFG_PATH,'w') as f: yaml.safe_dump(c, f, sort_keys=False, allow_unicode=True)
from src.utils import load_config
cfg = load_config('config.yaml')
print("RGB:", cfg['paths']['features_rgb'])
print("KP :", cfg['paths']['features_kp'])

# verify both stride-2 dirs are populated (guards against the earlier empty-dataset bug)
for lbl, pth in [('RGB',cfg['paths']['features_rgb']),('KP',cfg['paths']['features_kp'])]:
    d = os.path.join(pth,'train')
    print(f"{lbl}: files={len(os.listdir(d)) if os.path.isdir(d) else 0}")

In [ ]:
from src.train import run_training
r3 = run_training(cfg, ("rgb","kp"), "E3_fusion_s2", seed=42)
r4 = run_training(cfg, ("rgb","kp"), "E4_aux_s2",    seed=42, overrides={"model.aux_ctc": True})
print("\n=== STRIDE-2 (best dev WER) ===")
print(f"E1_s2 video:       51.28   (stride-4: 53.86)")
print(f"E3_s2 fusion:      {r3['best_dev_wer']:.2f}   (stride-4: 49.19)")
print(f"E4_s2 fusion+aux:  {r4['best_dev_wer']:.2f}   (stride-4: 46.03)")
print(f"E2 keypoint-only (unchanged bar): 44.06")

## Stage 10 — Test evaluation + artifacts (stride 2)
Score the stride-2 models on test, save the consolidated results, and produce the qualitative transcription demo and the resolution-study chart.

**Result: E4 stride-2 = 40.61 test WER**, beating keypoints-alone (44.71) by 4.10 points — fusion wins once resolutions are matched.

In [ ]:
import torch, os
from src.train import load_model_from_ckpt
from src.data import FeatureDataset, collate
from src.decode import evaluate_greedy
from src.vocab import load_vocab
from src.utils import load_json, p, get_device
from torch.utils.data import DataLoader

device = get_device()
g2i, i2g = load_vocab(p(cfg,'manifests')/'vocab.json')
RUNS = cfg['paths']['runs']
test_items = load_json(p(cfg,'manifests')/'test.json')

experiments = [("E1_rgb_s2",("rgb",)), ("E3_fusion_s2",("rgb","kp")), ("E4_aux_s2",("rgb","kp"))]

print(f"{'Exp':<16}{'Dev':<8}{'Test':<8}")
print("-"*32)
for run_name, streams in experiments:
    ckpt = os.path.join(RUNS, f"{run_name}_seed42", "best.pt")
    model, ckpt_cfg, _ = load_model_from_ckpt(ckpt, device)
    ds = FeatureDataset(ckpt_cfg, test_items, 'test', streams, g2i, train_mode=False)
    dl = DataLoader(ds, batch_size=16, collate_fn=collate)
    tw, _, _ = evaluate_greedy(model, dl, i2g, device)
    dev = torch.load(ckpt, map_location='cpu').get('dev_wer', float('nan'))
    print(f"{run_name:<16}{dev:<8.2f}{tw['wer']:<8.2f}")
print("\n(stride-4 test: E1 53.37 | E2 44.71 | E3 47.86 | E4 45.10)")

In [ ]:
import json, os
final = {
  "stride4": {"E1":53.37,"E2":44.71,"E3":47.86,"E4":45.10},
  "stride2": {"E1":50.55,"E3":44.37,"E4":40.61},
  "stride2_dev": {"E1":51.28,"E3":44.31,"E4":41.03},
  "headline": "E4 stride-2 fusion 40.61 test beats E2 keypoint-only 44.71 by 4.10"
}
path = os.path.join(cfg['paths']['runs'], 'final_results_stride2.json')
json.dump(final, open(path,'w'), indent=2)
print("saved:", path); print(json.dumps(final, indent=2))

In [ ]:
ds = FeatureDataset(ckpt_cfg, [test_items[0]], 'test', ("rgb","kp"), g2i, train_mode=False)
batch = collate([ds[0]])
out = model(batch, device)
print("type:", type(out))
if isinstance(out, dict):
    for k, v in out.items():
        print(f"  '{k}': {type(v).__name__}", tuple(v.shape) if hasattr(v,'shape') else v)
else:
    print("  shape:", tuple(out.shape))

In [ ]:
@torch.no_grad()
def decode_one(item):
    ds = FeatureDataset(ckpt_cfg, [item], 'test', ("rgb","kp"), g2i, train_mode=False)
    batch = collate([ds[0]])
    out = model(batch, device)
    logp = out["main"]                      # (T, B, classes) = (T, 1, 1232)
    ids = logp.argmax(-1).squeeze(1).cpu().tolist()   # (T,1)->(T,) list of class ids
    hyp, prev = [], None
    for c in ids:
        if c != 0 and c != prev:            # collapse repeats, drop blank(0)
            hyp.append(i2g.get(c, f"<{c}>"))
        prev = c
    return hyp

In [ ]:
import html, random

@torch.no_grad()
def decode_one(item):
    ds = FeatureDataset(ckpt_cfg, [item], 'test', ("rgb","kp"), g2i, train_mode=False)
    batch = collate([ds[0]])
    out = model(batch, device)
    logp = out["main"]                                  # (T, 1, 1232)
    ids = logp.argmax(-1).squeeze(1).cpu().tolist()
    hyp, prev = [], None
    for c in ids:
        if c != 0 and c != prev: hyp.append(i2g.get(c, f"<{c}>"))
        prev = c
    return hyp

random.seed(7)
sample_items = random.sample(test_items, min(12, len(test_items)))
rows_html, console = [], []
for it in sample_items:
    ref = it['glosses']; hyp = decode_one(it)
    ops, dist = align(ref, hyp); wer = 100.0*dist/max(len(ref),1)
    console += [f"\n[{it['id']}]  WER {wer:.0f}%", "  REF: "+" ".join(ref), "  HYP: "+" ".join(hyp)]
    cells=[]
    for op,r,h in ops:
        if op=='ok': cells.append(f'<span class="ok">{html.escape(h)}</span>')
        elif op=='sub': cells.append(f'<span class="sub" title="ref: {html.escape(r)}">{html.escape(h)}</span>')
        elif op=='ins': cells.append(f'<span class="ins">{html.escape(h)}</span>')
        elif op=='del': cells.append(f'<span class="del" title="deleted">{html.escape(r)}</span>')
    rows_html.append(f'<div class="card"><div class="meta"><span class="id">{html.escape(it["id"])}</span><span class="wer">WER {wer:.0f}%</span><span class="len">{len(ref)} glosses</span></div><div class="ref"><b>Reference:</b> {html.escape(" ".join(ref))}</div><div class="aligned"><b>Predicted:</b> {" ".join(cells)}</div></div>')

print("\n".join(console))

doc = f"""<!DOCTYPE html><html><head><meta charset="utf-8"><title>CSLR Transcription Demo</title>
<style>body{{font-family:-apple-system,Segoe UI,Roboto,sans-serif;max-width:900px;margin:24px auto;padding:0 16px;color:#1a1a1a;background:#fafafa}}h1{{font-size:20px;margin-bottom:2px}}.sub-h{{color:#666;font-size:13px;margin-bottom:18px}}.legend{{font-size:12px;margin:12px 0 20px;display:flex;gap:14px;flex-wrap:wrap}}.legend span{{padding:2px 8px;border-radius:4px}}.card{{background:#fff;border:1px solid #e3e3e3;border-radius:8px;padding:12px 14px;margin-bottom:12px}}.meta{{display:flex;gap:14px;font-size:12px;color:#777;margin-bottom:6px}}.meta .wer{{font-weight:600;color:#444}}.id{{font-family:monospace}}.ref{{font-size:13px;color:#555;margin-bottom:4px}}.aligned{{font-size:14px;line-height:1.9}}.ok{{color:#1a7f37}}.sub{{background:#fde2e2;color:#b91c1c;padding:1px 4px;border-radius:3px}}.ins{{background:#fff3cd;color:#92670a;padding:1px 4px;border-radius:3px;text-decoration:underline}}.del{{background:#e2e8f0;color:#475569;padding:1px 4px;border-radius:3px;text-decoration:line-through}}</style></head><body>
<h1>Continuous Sign Language Recognition — Transcription Demo</h1>
<div class="sub-h">Model: E4 (stride-2 video+keypoint fusion + auxiliary CTC) · PHOENIX-2014 test · greedy decoding · test WER 40.61%</div>
<div class="legend"><span class="ok">correct</span><span class="sub">substitution</span><span class="ins">insertion</span><span class="del">deletion</span></div>
{''.join(rows_html)}</body></html>"""
out_path = os.path.join(cfg['paths']['runs'], 'transcription_demo.html')
open(out_path,'w').write(doc)
print(f"\n\nHTML written to: {out_path}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np, os

labels   = ['E1\nvideo only', 'E3\nfusion', 'E4\nfusion+aux']
stride4  = [53.37, 47.86, 45.10]
stride2  = [50.55, 44.37, 40.61]
E2_BAR   = 44.71

x = np.arange(len(labels)); w = 0.36
fig, ax = plt.subplots(figsize=(8.6, 5.0), dpi=140)

b1 = ax.bar(x - w/2, stride4, w, label='stride 4 (coarse)',
            color='#c9d6e5', edgecolor='#5b7a99', linewidth=0.8)
b2 = ax.bar(x + w/2, stride2, w, label='stride 2 (fine)',
            color='#4a7fb5', edgecolor='#2f5174', linewidth=0.8)

# E2 reference line — label moved to LEFT side to avoid the -4.49
ax.axhline(E2_BAR, color='#b5462f', lw=1.6, ls='--', zorder=1)
ax.text(-0.45, E2_BAR + 0.45, f'E2 keypoint-only: {E2_BAR:.2f}',
        color='#b5462f', fontsize=9.5, ha='left', va='bottom', fontweight='bold')

for bars in (b1, b2):
    for r in bars:
        h = r.get_height()
        ax.text(r.get_x()+r.get_width()/2, h+0.4, f'{h:.2f}', ha='center', va='bottom', fontsize=9)

b2[2].set_color('#2e7d32'); b2[2].set_edgecolor('#1b4d1f')

# improvement arrows + drop labels (nudged up a bit so they clear the line)
for i in range(len(labels)):
    ax.annotate('', xy=(x[i]+w/2, stride2[i]), xytext=(x[i]-w/2, stride4[i]),
                arrowprops=dict(arrowstyle='->', color='#888', lw=1.0, alpha=0.7))
    drop = stride4[i]-stride2[i]
    ax.text(x[i]-w/2-0.02, stride4[i]+1.4, f'−{drop:.2f}',
            ha='left', fontsize=8.5, color='#555', style='italic')

ax.set_ylabel('Test WER (%)  ·  lower is better', fontsize=11)
ax.set_title('Effect of temporal resolution on fusion WER (PHOENIX-2014 test)\n'
             'At stride-2, fusion (E4) beats the best single stream',
             fontsize=12, fontweight='bold', pad=12)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(35, 58); ax.set_xlim(-0.7, 2.7)
ax.legend(loc='upper right', frameon=False, fontsize=9.5)
ax.spines[['top','right']].set_visible(False)
ax.grid(axis='y', alpha=0.25, lw=0.6)

# green win annotation — repositioned to sit left of the green bar, fully inside
ax.annotate('fusion now beats\nkeypoints-alone',
            xy=(x[2]+w/2, 40.61), xytext=(x[2]-0.55, 38.0),
            fontsize=9, color='#2e7d32', fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='#2e7d32', lw=1.2))

plt.tight_layout()
out_path = os.path.join(cfg['paths']['runs'], 'resolution_study.png')
plt.savefig(out_path, bbox_inches='tight', dpi=140)
plt.show()
print("saved:", out_path)